In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob, os, yaml, sparse, itertools, subprocess, sys, pickle

from sklearn.linear_model import ElasticNet, ElasticNetCV, Ridge, RidgeCV, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from Bio import SeqIO, Seq
import statsmodels.stats.api as sm

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "utils"))
from inSilicoMut_utils import *
from data_utils import *

h37Rv_path = "/n/data1/hms/dbmi/farhat/Sanjana/H37Rv"
h37Rv_seq = SeqIO.read(os.path.join(h37Rv_path, "GCF_000195955.2_ASM19595v2_genomic.gbff"), "genbank")
h37Rv_genes = pd.read_csv(os.path.join(h37Rv_path, "mycobrowser_h37rv_genes_v4.csv"))
h37Rv_coords = pd.read_csv(os.path.join(h37Rv_path, "h37Rv_coords_to_gene.csv"))
h37Rv_coords_dict = dict(zip(h37Rv_coords["pos"].values, h37Rv_coords["region"].values))

data_path = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs"

who_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/WHO_catalog_clean.csv")
who_variants["gene"] = [val.split("_")[0] for val in who_variants.mutation.values]

solo_results = pd.read_excel("/home/sak0914/who-analysis/data/SOLO primary_STATA_ver18Feb2023.xlsx", sheet_name=None)["Sheet1"]
solo_results["gene"] = [val.split("_")[0] for val in solo_results.variant.values]

drug_gene_mapping = pd.read_csv("/home/sak0914/who-analysis/data/drug_gene_mapping.csv")

cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/criticalConcentrations_updated.csv")

In [2]:
who_variants.drug.unique()

array(['STM', 'PZA', 'KAN', 'RIF', 'EMB', 'CAP', 'INH', 'ETH', 'LZD',
       'MXF', 'LEV', 'AMI', 'DLM', 'CFZ', 'BDQ'], dtype=object)

In [3]:
who_variants.confidence.unique()

array(['1) Assoc w R', '2) Assoc w R - Interim',
       '3) Uncertain significance', '4) Not assoc w R - Interim',
       '5) Not assoc w R'], dtype=object)

In [4]:
drug = "KAN"
who_variants.query("drug==@drug & confidence=='1) Assoc w R'")

,drug,genome_index,confidence,mutation,gene
40,KAN,2715344,1) Assoc w R,eis_c.-12C>T,eis
159,KAN,2715339,1) Assoc w R,eis_c.-8delC,eis
161,KAN,2715342,1) Assoc w R,eis_c.-10G>A,eis
162,KAN,2715369,1) Assoc w R,eis_c.-37G>T,eis
164,KAN,1473329,1) Assoc w R,rrs_n.1484G>T,rrs
185,KAN,2715346,1) Assoc w R,eis_c.-14C>T,eis
188,KAN,1473246,1) Assoc w R,rrs_n.1401A>G,rrs


In [5]:
drug = "AMI"
who_variants.query("drug==@drug & confidence=='1) Assoc w R'")

,drug,genome_index,confidence,mutation,gene
187,AMI,2715346,1) Assoc w R,eis_c.-14C>T,eis
190,AMI,1473246,1) Assoc w R,rrs_n.1401A>G,rrs


In [6]:
drug = "CAP"
who_variants.query("drug==@drug & confidence=='1) Assoc w R'")

,drug,genome_index,confidence,mutation,gene
64,CAP,1918647,1) Assoc w R,tlyA_p.Asn236Lys,tlyA
65,CAP,1918160,1) Assoc w R,tlyA_p.Leu74Pro,tlyA
163,CAP,1473247,1) Assoc w R,rrs_n.1402C>T,rrs
166,CAP,1473329,1) Assoc w R,rrs_n.1484G>T,rrs
189,CAP,1473246,1) Assoc w R,rrs_n.1401A>G,rrs


In [8]:
h37Rv_genes.query("Symbol=='eis'")

,Gene_Ind,Feature,Start,End,Strand,Frame,H37rv_GeneID,Symbol,Function,Product,...,SWISS-MODEL,Orthologues M. leprae,Orthologues M. marinum,Orthologues M. smegmatis,Orthologues M. bovis,Orthologues M. lepromatosis,Orthologues M. tuberculosis,Orthologues M. abscessus,Orthologues M. haemophilum,Orthologues M. orygis
2484,2484,CDS,2714124,2715332,-,0.0,Rv2416c,eis,"Acetylation, substrate unknown. Involved in in...","Enhanced intracellular survival protein Eis, G...",...,P9WFK7,NaN,MMAR_3740,MSMEG_3513,Mb2439c,NaN,NaN,NaN,NaN,NaN


In [2]:
isolate_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/isolate_variants.tsv", sep="\t")
len(isolate_variants.Isolate.unique())

6953

In [3]:
isolate_variants.FILTER.unique()

array(['PASS', 'Del;LowCov', 'Amb', 'Del', 'LowCov', 'Amb;LowCov',
       'Del;Amb', 'Del;Amb;LowCov'], dtype=object)

In [4]:
isolate_variants

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate
0,4013,T,C,PASS,1.0,recF,missense_variant,c.734T>C,p.Ile245Thr,00R0223
1,7362,G,C,PASS,1.0,gyrA,missense_variant,c.61G>C,p.Glu21Gln,00R0223
2,8048,C,A,PASS,1.0,gyrA,synonymous_variant,c.747C>A,p.Gly249Gly,00R0223
3,11879,A,G,PASS,1.0,Rv0008c,missense_variant,c.433T>C,p.Ser145Pro,00R0223
4,12984,C,T,PASS,1.0,ppiA,missense_variant,c.517C>T,p.Pro173Ser,00R0223
...,...,...,...,...,...,...,...,...,...,...
12256616,4400039,T,C,PASS,1.0,Rv3910,missense_variant,c.3443T>C,p.Val1148Ala,ERR4828566
12256617,4400660,AC,A,PASS,1.0,sigM,frameshift_variant,c.478delC,p.Arg160fs,ERR4828566
12256618,4407588,T,C,PASS,1.0,gid,synonymous_variant,c.615A>G,p.Ala205Ala,ERR4828566
12256619,4408002,ATCCACGACCC,A,PASS,0.98,gid,frameshift_variant,c.191_200delGGGTCGTGGA,p.Arg64fs,ERR4828566


In [35]:
isolate_variants.loc[isolate_variants["EFFECT"].str.contains('|'.join(no_AA_change))]

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate,mutation
2,8048,C,A,PASS,1.0,gyrA,synonymous_variant,c.747C>A,p.Gly249Gly,00R0223,gyrA_c.747C>A
8,26959,C,G,PASS,1.0,Rv0021c-whiB5,intergenic_region,n.26959C>G,.,00R0223,Rv0021c-whiB5_n.26959C>G
9,28640,T,C,PASS,1.0,Rv0024,synonymous_variant,c.279T>C,p.Val93Val,00R0223,Rv0024_c.279T>C
10,34044,T,C,PASS,1.0,Rv0030-bioF2,intergenic_region,n.34044T>C,.,00R0223,Rv0030-bioF2_n.34044T>C
12,42967,G,C,PASS,1.0,mtc28,synonymous_variant,c.399C>G,p.Pro133Pro,00R0223,mtc28_c.399C>G
...,...,...,...,...,...,...,...,...,...,...,...
487974,4356110,G,C,PASS,1.0,eccD1,synonymous_variant,c.1104G>C,p.Leu368Leu,C050,eccD1_c.1104G>C
487975,4366272,G,C,PASS,1.0,eccA2,synonymous_variant,c.567C>G,p.Ala189Ala,C050,eccA2_c.567C>G
487976,4367478,C,G,PASS,1.0,eccE2,synonymous_variant,c.1044G>C,p.Ala348Ala,C050,eccE2_c.1044G>C
487980,4382054,T,C,PASS,1.0,Rv3896c,synonymous_variant,c.798A>G,p.Ala266Ala,C050,Rv3896c_c.798A>G


# Read in Elastic Net Model Fit on `original_train_set` only

In [23]:
drug = "PZA"
scaler = StandardScaler()

regression_matrix = pd.read_pickle(os.path.join(data_path, drug, "whole_genome/regression_matrix_no_drug_res.pkl"))
pca_df = pd.read_csv(os.path.join(data_path, drug, "whole_genome/PCA_eigenvec_df.csv"), index_col=0)

df_phenos = pd.read_csv(os.path.join(data_path, drug, "data_for_model.csv"), index_col=0)

# elastic_net_model = pickle.load(open(os.path.join(data_path, drug, "whole_genome/model_no_drug_res.sav"), 'rb'))
elastic_net_model = pickle.load(open(os.path.join(data_path, drug, "whole_genome/full_feature_select.sav"), 'rb'))
print(f"alpha = {elastic_net_model.alpha_}, L1 = {elastic_net_model.l1_ratio_}")
coef_df = pd.DataFrame({"feature": list(regression_matrix.columns) + [f"PC{num}" for num in range(12)], "coef": np.squeeze(elastic_net_model.coef_)})

alpha = 0.1, L1 = 0.4


In [18]:
df_phenos["Lineage"].value_counts()

Lineage
4    1078
2     112
Name: count, dtype: int64

In [4]:
df_phenos.groupby("Lineage")[["Binary", f"{drug}_midpoint"]].mean()

,Binary,PZA_midpoint
Lineage,,
2,0.339286,167.633929
4,0.442486,198.140074


# Extract features with `coef != 0` (relevant by the L1 part of the model)

In [24]:
# get only significant features by permutation test, them perform bootstrapping to get confidence intervals
# sig_features = coef_df.query("Bonferroni_pval < 0.05").feature.values

# get only features where coef != 0. These are shrunk to 0 by the LASSO part of the Elastic Net model
sig_features = coef_df.query("coef != 0").feature.values

if len(coef_df.query("coef != 0 & feature.str.contains('PC')")) == 0:
    include_PC = False
else:
    include_PC = True

print(f"{len(sig_features)} significant features, include principal components = {include_PC}")
# features significant by permutation test are a subset of those with nonzero coefficients
# assert len(set(coef_df.query("Bonferroni_pval < 0.05").feature.values) - set(sig_features)) == 0

regression_matrix_downselect = regression_matrix[sig_features]
assert regression_matrix_downselect.shape[1] == len(sig_features)

4141 significant features, include principal components = False


# Build a Ridge regression model using downselected features

In [31]:
ridge_model = RidgeCV(alphas=np.logspace(-5, 5, 11))

if include_PC:
    X_train = scaler.fit_transform(pd.concat([regression_matrix_downselect.loc[df_phenos.query("category=='original_train_set'").index.values],
                                              pca_df.loc[df_phenos.query("category=='original_train_set'").index.values]
                                             ], axis=1).values
                                  )
    
    X_test = scaler.fit_transform(pd.concat([regression_matrix_downselect.loc[df_phenos.query("category=='original_test_set'").index.values],
                                             pca_df.loc[df_phenos.query("category=='original_test_set'").index.values]
                                            ], axis=1).values
                                  )
    X = scaler.fit_transform(pd.concat([regression_matrix_downselect.loc[df_phenos.index.values],
                                        pca_df.loc[df_phenos.index.values]
                                        ], axis=1).values
                                  )
else:
    X_train = scaler.fit_transform(regression_matrix_downselect.loc[df_phenos.query("category=='original_train_set'").index.values].values)
    
    X_test = scaler.fit_transform(regression_matrix_downselect.loc[df_phenos.query("category=='original_test_set'").index.values].values)

    X = scaler.fit_transform(regression_matrix_downselect.loc[df_phenos.index.values].values)

y_train = np.log2(df_phenos.query("category=='original_train_set'")[f"{drug}_midpoint"].values)
y_test = np.log2(df_phenos.query("category=='original_test_set'")[f"{drug}_midpoint"].values)
y = np.log2(df_phenos[f"{drug}_midpoint"].values)

100.0


# Get predictions on the test dataset and see how well the model does

In [32]:
ridge_model.fit(X_train, y_train)
print(ridge_model.alpha_)

# use the model trained on the training data to get predictions on the training data only
y_pred = ridge_model.predict(X_test)
pred_df = df_phenos[["category", f"{drug}_lower_bound", f"{drug}_upper_bound"]].loc[regression_matrix_downselect.index.values].query("category=='original_test_set'").reset_index()
pred_df["y_test"] = y_test
pred_df["y_pred"] = y_pred
pred_df["y_test_exp"] = np.exp2(pred_df["y_test"])
pred_df["y_pred_exp"] = np.exp2(pred_df["y_pred"])

within_doubling = len(pred_df.query(f"{drug}_lower_bound / 2 <= y_pred_exp <= {drug}_upper_bound * 2")) / len(pred_df)

print(f"MAE: {np.mean(np.abs(y_pred - y_test))}, MSE: {np.mean(np.exp2(y_pred - y_test))}, EA: {within_doubling}")

MAE: 1.1167378873072742, MSE: 1.5286916819182876, EA: 0.6680672268907563


In [33]:
ridge_model.fit(X, y)
print(ridge_model.alpha_)

# use the model trained on the full data to get predictions on the training data only
y_pred = ridge_model.predict(X_test)
pred_df = df_phenos[["category", f"{drug}_lower_bound", f"{drug}_upper_bound"]].loc[regression_matrix_downselect.index.values].query("category=='original_test_set'").reset_index()
pred_df["y_test"] = y_test
pred_df["y_pred"] = y_pred
pred_df["y_test_exp"] = np.exp2(pred_df["y_test"])
pred_df["y_pred_exp"] = np.exp2(pred_df["y_pred"])

within_doubling = len(pred_df.query(f"{drug}_lower_bound / 2 <= y_pred_exp <= {drug}_upper_bound * 2")) / len(pred_df)

print(f"MAE: {np.mean(np.abs(y_pred - y_test))}, MSE: {np.mean(np.exp2(y_pred - y_test))}, EA: {within_doubling}")

100.0
MAE: 0.7241207899566382, MSE: 1.145057457910479, EA: 0.8445378151260504


In [5]:
#pred_df.query(f"~({drug}_lower_bound / 2 <= y_pred_exp <= {drug}_upper_bound * 2) & ({drug}_upper_bound != 500000)")

# Permutation Test to Find Significant Features

In [22]:
# use the regularization parameter determined above
def perform_permutation_test(model, X, y, num_reps, progress_bar=False):

    reg_param = model.alpha_
            
    coefs = []    
    for i in range(num_reps):

        if i == 0:
            print(f"Fitting {num_reps} permuted Ridge models using regularization parameter = {reg_param}")
            
        # shuffle phenotypes. np.random.shuffle works in-place
        y_permute = y.copy()
        np.random.shuffle(y_permute)

        rep_model = Ridge(alpha=reg_param, max_iter=100000)
        rep_model.fit(X, y_permute)
        coefs.append(np.squeeze(rep_model.coef_))
        
        if progress_bar:
            if i % int(num_reps / 10) == 0:
                print(i)
        
    return pd.DataFrame(coefs)
    
    
def add_pvalues(drug, matrix, model, permutation_df, feat_array):

    coef_df = pd.DataFrame({"feature": feat_array, "coef": np.squeeze(model.coef_)})
    coef_df["indel"] = [0 if "_" in val and "NC" not in val else 1 for val in coef_df.feature.values]
    # coef_df["WHO_cat1"] = coef_df["feature"].map(dict(zip(who_variants.query("drug!=@drug")["mutation"], who_variants.query("drug!=@drug")["confidence"])))
    coef_df["WHO_cat1"] = coef_df["feature"].map(dict(zip(who_variants["mutation"], who_variants["confidence"])))
    coef_df["gene"] = ["_".join(val.split("_")[:-1]) if "_" in val else val for val in coef_df.feature.values]

    for i, row in coef_df.iterrows():
        # p-value is the proportion of permutation coefficients that are AT LEAST AS EXTREME as the test statistic
        # ONE-SIDED because we are interested in the sign of the coefficient
        if row["coef"] > 0:
            coef_df.loc[i, "pval"] = np.mean(permutation_df[row["feature"]] >= row["coef"])
        else:
            coef_df.loc[i, "pval"] = np.mean(permutation_df[row["feature"]] <= row["coef"])
            
    _, bh_pvals, _, _ = sm.multipletests(coef_df["pval"], method='fdr_bh', is_sorted=False, returnsorted=False)
    _, bonferroni_pvals, _, _ = sm.multipletests(coef_df["pval"], method='bonferroni', is_sorted=False, returnsorted=False)

    coef_df["BH_pval"] = bh_pvals
    coef_df["Bonferroni_pval"] = bonferroni_pvals
    
    return coef_df

In [25]:
# ridge_permutation_df = perform_permutation_test(ridge_model, X, y, 1000, progress_bar=True)

if include_PC:
    feat_array = np.concatenate([sig_features, pca_df.columns])
else:
    feat_array = np.copy(sig_features)

# ridge_permutation_df.columns = feat_array
# ridge_permutation_df.to_csv(os.path.join(data_path, drug, "whole_genome/permutation_df_ridge.csv"), index=False)
ridge_permutation_df = pd.read_csv(os.path.join(data_path, drug, "whole_genome/permutation_df_ridge.csv"))

In [26]:
# ridge_coef_df = add_pvalues(drug, regression_matrix_downselect, ridge_model, ridge_permutation_df, feat_array)
# ridge_coef_df.to_csv(os.path.join(data_path, drug, "whole_genome/ridge_results.csv"), index=False)
ridge_coef_df = pd.read_csv(os.path.join(data_path, drug, "whole_genome/ridge_results.csv"))

In [41]:
ridge_coef_df.query("pval < 0.05 & indel==1 & ~gene.str.contains('|'.join(['PPE', 'PGRS'])) & (coef > 0.1 | coef < -0.1)").sort_values("coef", ascending=False)

,feature,coef,indel,WHO_cat1,gene,pval,BH_pval,Bonferroni_pval
4120,pncA,0.527182,1,NaN,pncA,0.000,0.000000,0.0
3993,Rv2041c,0.137648,1,NaN,Rv2041c,0.023,0.378973,1.0
3927,Rv0196,-0.100226,1,NaN,Rv0196,0.046,0.378973,1.0
3881,NC_2854,-0.128827,1,NaN,NC,0.049,0.378973,1.0


In [48]:
h37Rv_coords.query("region=='NC_2854'")

,pos,region
4156729,4156730,NC_2854
4156730,4156731,NC_2854
4156731,4156732,NC_2854
4156732,4156733,NC_2854
4156733,4156734,NC_2854
...,...,...
4156975,4156976,NC_2854
4156976,4156977,NC_2854
4156977,4156978,NC_2854
4156978,4156979,NC_2854


In [ ]:
Rv3712 (upstream, ligase), Rv2041c (sugar transport), Rv0196 (transcriptional regulatory protein), bacA

In [37]:
solo_results.query("drug=='Pyrazinamide' & Initial_Confidence_Grading in ['1) Assoc w R', '2) Assoc w R - Interim']").gene.unique()

array(['pncA'], dtype=object)

In [40]:
#solo_results.query("drug=='Pyrazinamide' & Initial_Confidence_Grading in ['1) Assoc w R', '2) Assoc w R - Interim']")

In [46]:
ridge_coef_df.query("pval < 0.05 & indel==0 & ~gene.str.contains('|'.join(['PPE', 'PGRS']))  & (coef > 0.1 | coef < -0.1)").sort_values("coef", ascending=False)#.head(20)

,feature,coef,indel,WHO_cat1,gene,pval,BH_pval,Bonferroni_pval
2264,bacA_p.Trp627Arg,0.200912,0,NaN,bacA,0.000,0.000000,0.0
3463,pncA_p.Gln10Arg,0.197677,0,1) Assoc w R,pncA,0.020,0.378973,1.0
3443,pncA_c.309C>G,0.154411,0,NaN,pncA,0.006,0.211584,1.0
3459,pncA_p.Asp49Asn,0.135162,0,2) Assoc w R - Interim,pncA,0.034,0.378973,1.0
3501,pncA_p.Thr76Pro,0.123550,0,1) Assoc w R,pncA,0.036,0.378973,1.0
3465,pncA_p.Gln141Pro,0.101579,0,1) Assoc w R,pncA,0.035,0.378973,1.0


In [52]:
significant_SNP_df = pd.DataFrame(ridge_coef_df.query("pval < 0.05 & indel==0 & ~gene.str.contains('|'.join(['PPE', 'PGRS']))").groupby("gene")["coef"].count())
significant_SNP_df.query("coef > 1").sort_values("coef", ascending=False)

,coef
gene,
pncA,11
Rv3467,5
Rv1199c,3
Rv0485,2
Rv0797,2
Rv1313c,2
Rv2752c,2
Rv3524,2
Rv3813c,2


In [50]:
ridge_coef_df.query("pval < 0.05 & indel==0 & gene in ['Rv3467', 'Rv1199c']")

,feature,coef,indel,WHO_cat1,gene,pval,BH_pval,Bonferroni_pval
1039,Rv1199c_p.Ala168Thr,0.010598,0,NaN,Rv1199c,0.012,0.314506,1.0
1050,Rv1199c_p.Ile139Ser,0.013026,0,NaN,Rv1199c,0.000,0.000000,0.0
1051,Rv1199c_p.Ile281Thr,0.003930,0,NaN,Rv1199c,0.004,0.186112,1.0
1979,Rv3467_p.Ala214Val,0.027496,0,NaN,Rv3467,0.028,0.378973,1.0
1980,Rv3467_p.Asn222Thr,0.017846,0,NaN,Rv3467,0.016,0.378973,1.0
1981,Rv3467_p.Asp232His,0.017846,0,NaN,Rv3467,0.016,0.378973,1.0
1982,Rv3467_p.Cys226Phe,0.017846,0,NaN,Rv3467,0.016,0.378973,1.0
1987,Rv3467_p.Tyr236His,0.017846,0,NaN,Rv3467,0.016,0.378973,1.0


In [77]:
# length ~1.05 million
cetr = pd.read_csv("/home/sak0914/CETR_isolate_details.csv")
cetr.shape

(1048575, 44)

In [88]:
pza_df = cetr.loc[~pd.isnull(cetr["PZA"])].reset_index(drop=True)
pza_df.shape

(1266, 44)

In [134]:
count_missing = 0
missing_sample_idx = []

for i, row in pza_df.iterrows():

    sample_ids_lst = [row["ID"], row["WGSAccessionNumber/Run Accession"], row["BioSample"], row["Alt ID"]]
    found = False
    
    for sample_id in sample_ids_lst:

        if not pd.isnull(sample_id):
            
            fNames = [f"/n/scratch3/users/s/sak0914/annotated_VCF/{sample_id.replace('-', '')}.eff.vcf", 
                      f"/n/scratch3/users/s/sak0914/annotated_VCF/{sample_id.replace('-', '_')}.eff.vcf",
                      f"/n/scratch3/users/s/sak0914/annotated_VCF/{sample_id.replace('.00', '')}.eff.vcf",
                      f"/n/scratch3/users/s/sak0914/annotated_VCF/{sample_id.replace('.0', '')}.eff.vcf"
                     ]
    
            for fName in fNames:
                if os.path.isfile(fName):
                    found = True
                    break
                else:
                    pass
                
    if not found:
        count_missing += 1
        missing_sample_idx.append(i)

# the original PZA sample count was 1250, so I think the 16 missing samples failed QC or something, so we don't have the .vcf.gz files for them
print(f"{count_missing} missing samples")

16 missing samples


In [56]:
vcf_eff = glob.glob("/n/scratch3/users/s/sak0914/annotated_VCF/*.eff.vcf")
vcfs = glob.glob("/n/scratch3/users/s/sak0914/annotated_VCF/*.vcf")
set(vcfs) - set(vcf_eff)

set()

In [44]:
h37Rv_genes.query("Symbol=='pca'")

,Gene_Ind,Feature,Start,End,Strand,Frame,H37rv_GeneID,Symbol,Function,Product,...,SWISS-MODEL,Orthologues M. leprae,Orthologues M. marinum,Orthologues M. smegmatis,Orthologues M. bovis,Orthologues M. lepromatosis,Orthologues M. tuberculosis,Orthologues M. abscessus,Orthologues M. haemophilum,Orthologues M. orygis
3048,3048,CDS,3319663,3323046,-,0.0,Rv2967c,pca,Involved in gluconeogenesis and lipogenesis. P...,Probable pyruvate carboxylase Pca (pyruvic car...,...,I6YEU0,"ML1665,ML1665c",MMAR_1749,MSMEG_2412,Mb2991c,NaN,NaN,NaN,NaN,NaN
